# 实践一:简单计算图

**计算图:**
从左往右计算(出发点到结束点)--正向传播;反之--反向传播
特点:局部计算
优点:可以通过反向传播高效计算导数--方式:沿与正方向相反方向,乘上局部倒数
链式法则:某函数由复合函数表示,该复合函数的导数可以用构成复合函数的各个函数的导数乘积表示(可以相互抵消)

## 1.画出计算图
u=x*y  v=x*x  w=u+v  f=w*w
文字说明

```mermaid  
%% 开始用来渲染
graph LR  
%%left-right
    x((x)) -->|乘 y| u    
    %% (())定义一个圆形节点,x1是节点的内部代号,x2是圆圈里实际显示出来的文字;画箭头对应u=x*y
    y((y)) -->|乘 x| u    
    %% 同理,对应y的参与u=x*y
    x -->|平方| v   
    %% 圆x分出新去向:v=x*x
    u -->|加 v| w   
    %% 圆u分出新去向:w=u+v
    v -->|加 u| w   
    %% 圆v分出新去向:w=v+u
    w -->|平方| f   
    %% 圆w分出新去向:f=w*w
```
**注释必须独占一行**


## 2. NumPy 代码,按照计算图从后向前计算梯度

In [1]:
import numpy as np
# 设定初始值
x = 2.0
y = 3.0

# 前向传播(计算途中从左往右)
u = x * y
v = x ** 2
w = u + v
f = w ** 2

# 反向传播（手动算梯度）
grad_f = 1.0    # 反向传播起点,从f开始往回走,f对自己的导数是1
grad_w = grad_f * (2 * w)   # f对w的导数 f=w*w--2w
grad_u = grad_w * 1.0   # f对u的导数
grad_v = grad_w * 1.0   # f对v的导数---把梯度从w传给u,v

grad_x_from_u = grad_u * y  # u=x*y 对x求导就是y 
grad_y = grad_u * x     # u对y求导是x----以上两步是求x和y梯度的第一步
grad_x_from_v = grad_v * (2 * x) # 求x梯度的第二条路经

grad_x = grad_x_from_u + grad_x_from_v # 因为在计算图中x分叉,所以在反向传播中,两条路径梯度要相加

print(f"NumPy手动结果: df/dx = {grad_x}, df/dy = {grad_y}")

NumPy手动结果: df/dx = 140.0, df/dy = 40.0


In [2]:
import torch
x_t=torch.tensor(2.0,requires_grad=True)
y_t=torch.tensor(3.0,requires_grad=True) 
# 把x,y转换成张量_t用来提醒是张量;requires_grad=True:告诉PyTorch,注意这两个变量,参与计算后要知晓他们的梯度

f_t=(x_t * y_t + x_t ** 2)**2
# 计算公式,PyTorch底层自动建立计算图

f_t.backward()
# 沿着 PyTorch 底层自动建立的计算图,把所有局部导数乘起来,有分叉则自动相加,梯度结果存进x_t和y_t的.grad属性里

print(f"PyTorch结果: df/dx = {x_t.grad.item()}, df/dy = {y_t.grad.item()}")
# 取出梯度(张量),把张量中的具体数值提取出来,变成浮点数,方便打印比较

PyTorch结果: df/dx = 140.0, df/dy = 40.0


## 3. PyTorch 的 autograd 检查自己的结果是否一致

In [3]:
import torch

print("\n--- PyTorch 验证 ---")

x_t=torch.tensor(x, requires_grad=True)
y_t=torch.tensor(y, requires_grad=True)
# 把x和y转换成PyTorch张量,并开启梯度追踪

f_t=(x_t * y_t + x_t ** 2) ** 2
# 前向传播

f_t.backward()
# 反向传播
 
print(f"PyTorch自动结果: df/dx = {x_t.grad.item()}, df/dy = {y_t.grad.item()}")
# 打印 PyTorch 算出的梯度

if abs(grad_x-x_t.grad.item()) < 1e-6 and abs(grad_y - y_t.grad.item()) < 1e-6:
    print("\n结论:NumPy手算与PyTorch自动求导一致")
else:
   print("\n结论:结果不一致请,检查公式")
# 验证是否一致 1e-6:极小的阈值


--- PyTorch 验证 ---
PyTorch自动结果: df/dx = 140.0, df/dy = 40.0

结论:NumPy手算与PyTorch自动求导一致


# 实践二:神经网络的梯度
Task1中的make_moons数据中取,完成一次前向传播和反向传播

**Task1中的make_moons**

In [7]:
import numpy as np #用来做数字计算
from sklearn.datasets import make_moons #导入函数,生成月牙型状数据
from sklearn.model_selection import train_test_split #划分数据集工具:训练验证和测试集
from sklearn.preprocessing import StandardScaler #导入标准化工具,缩放数据,便于训练

# 固定随机种子
SEED = 42
def load_moons_data():
    np.random.seed(SEED) #给np设置常用种子,只要种子固定生成的随机数相同

# 生成月牙数据
    X, y = make_moons(n_samples=1500, noise=0.2, random_state=SEED)
# 总共生成1500个数据点;给点加一点噪声,模拟真实世界数据;固定随机种子,每次生成月牙点一样
# 返回两个结果:x每个点的坐标,是1500组(x,y);y每个点的标签(0/1)

# 划分数据集
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=SEED)
    X_val, X_test,y_val,  y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)
# 全部数据的30%到临时变量X_temp, y_temp  剩下70%到训练集X_train,y_train;固定切分,每次切出来的数据不变
# 将刚才30%的临时数据对半,一般到验证集X_val,y_val,一半到测试机
# 标准化(保留一份未标准化的副本用于)
    scaler = StandardScaler() #创建标准化工具对象,准备缩放数据
    X_train_scaled = scaler.fit_transform(X_train) #fit:只用训练集计算均值,方差 transform:用算出的均值方差缩放训练集坐标
    X_val_scaled = scaler.transform(X_val)#验证集制作transform缩放,不重新fit,沿用训练集的
    X_test_scaled = scaler.transform(X_test)#测试集同样

    return {
        'X': X, 'y': y,                        # 原始数据（未标准化）
        'X_train': X_train, 'y_train': y_train,  # 划分后未标准化
        'X_train_scaled': X_train_scaled,'X_val': X_val_scaled, 'y_val': y_val,  # 标准化后
        'X_test': X_test_scaled, 'y_test': y_test,
        'scaler': scaler
            }

## 1. 确定隐藏层神经元数量,初始化W1,b1,W2,b2

In [ ]:
import numpy as np
data = load_moons_data()
X = data['X_train_scaled'][0:1]         
# 标准化后的训练集里取第0号样本([0:1]是为了保留样本维度即1个样本2给特征)
y = data['y_train'][0:1].reshape(-1, 1)  
# 取对应标签,reshape成列向量(一维--二维,和输出一样)

np.random.seed(42) #固定随机种子
W1 = np.random.randn(2, 4) * 0.1   
# 第一层权重,形状(2,4)的标准正态分布矩阵(输入2维,隐藏层4维),0.1使数值缩小,避免梯度爆炸
b1 = np.zeros((1, 4))    
# 第一层偏置,全0(权重已经随机了,神经元之间不会对称),因为输出4个神经元,所以b1有四个数,形状(1,4),方便后面广播相加
W2 = np.random.randn(4, 1) * 0.1    
# 第二层权重,形状(4,1)隐藏层4维,输出1维
b2 = np.zeros((1, 1))               
# 第二层偏置

## 2. 前向传播

In [ ]:
Z1 = X @ W1 + b1                    
# 第一层线性变换(1,2)矩阵乘法(2,4)+(1,4)=(1,4)(未激活)
A1 = np.maximum(0, Z1)              
# ReLU激活 引入非线性(负数-0,正数不变)

Z2 = A1 @ W2 + b2                   
# 第二层线性变换(1,4)矩阵乘法(4,1)=(1,1)
A2 = 1 / (1 + np.exp(-Z2))          
# Sigmoid激活 压到(0,1),表示属于类别一的概率
A2c = np.clip(A2, 1e-15, 1 - 1e-15)
# A2里面的数值限制在区间,结果赋值给A2c(clipped);1e-15科学计数法10的-15次方(很小),如果是0后续训练不了ln0=-lnf
Loss = -(y * np.log(A2c) + (1 - y) * np.log(1 - A2c))
# 二元交叉熵损失L=-[ylogA2+(1-y)log(1-A2)] y=0/1


## 3. 计算图,链式法则求梯度
```mermaid
graph LR
    X[X] --> Z1[Z1]
    W1[W1] --> Z1
    b1[b1] --> Z1
    Z1 --> A1[A1]
    A1 --> Z2[Z2]
    W2[W2] --> Z2
    b2[b2] --> Z2
    Z2 --> A2[A2]
    A2 --> L[Loss]
    y[y] --> L
```


-->表示数据流向;[x]:方括号代表方形节点,x是在图上显示文字;已有节点再出现时不用括号

In [ ]:
dA2 = -(y / A2c) + (1 - y) / (1 - A2c)   
# Loss对A2的偏导 (1,1)
dZ2 = dA2 * A2 * (1 - A2)
# 链式法则:Loss对Z2的偏导:dA2*Sigmoid的导数(A2(1-A2))  偏导L/Z2=L/A2*A2/Z2 (1,1)
dW2 = A1.T @ dZ2                         
# Loss对W2的偏导 A1.T:转置(1,4)--(4,1) Z2=A1W2+b2,对W2求导得A1.T@偏导L/Z2 (4,1)
db2 = np.sum(dZ2, axis=0, keepdims=True)
# Loss对b2偏导 sum:b2对每个样本都贡献一份梯度，要所有的梯度(这里只有一个样本不变,后面改成batch时使用)axis:沿样本维度求和 keepdims=True:保持二维形状,与b2(1,1)匹配
dA1 = dZ2 @ W2.T
# Loss对A1的偏导 Z2=A1W2+b2-L/Z2*W2.T (4,1)--(1,4)  (1,1)@(1,4)=(1,4)
dZ1 = dA1 * (Z1 > 0)
# Loss对Z1的偏导,穿过了ReLU激活函数(正数/0) (Z1>0:得布尔矩阵-转成0/1)
dW1 = X.T @ dZ1
# Loss对W1的偏导,Z1=XW1+b1--X.T*偏导L/Z1 (1,2)--(2,1) (2,1)@(1,4)=(2,4)形状和W1一致
db1 = np.sum(dZ1, axis=0, keepdims=True)
# Loss对b1的偏导,沿样本维度求和,保持二维(同上) (1,4)

## 4. PyTorch自动求导

In [ ]:
import torch
X_t = torch.tensor(X, dtype=torch.float32)
# NumPy数组X转成PyTorch张量(常量不需要梯度)
y_t = torch.tensor(y, dtype=torch.float32)
# 标签转为张量
W1_t = torch.tensor(W1, dtype=torch.float32, requires_grad=True)
# W1转为张量,开启梯度追踪,之后会自动用链式法则
b1_t = torch.tensor(b1, dtype=torch.float32, requires_grad=True)
# 同理,b1,W2,b2
W2_t = torch.tensor(W2, dtype=torch.float32, requires_grad=True)
b2_t = torch.tensor(b2, dtype=torch.float32, requires_grad=True)
Z1_t = X_t @ W1_t + b1_t
# PyTorch版本的第一层线性变换 张量也支持矩阵乘法(1,4)
A1_t = torch.relu(Z1_t)
# ReLU激活
Z2_t = A1_t @ W2_t + b2_t
# 第二层
A2_t = torch.sigmoid(Z2_t)
# Sigmoid激活(1,1)
Loss_t = torch.nn.functional.binary_cross_entropy(A2_t, y_t)
# PyTorch内置的BCE损失
Loss_t.backward()
# 反向传播


## 5. 对比

In [ ]:
def compare(name, np_g, t_g):
# 定义对比函数 np_g:NumPy手算梯度,t_g:PyTorch自动计算梯度
    print(f"{name}")
    print(f"形状: NumPy {np_g.shape}, Torch {t_g.shape}")
# 打印形状,看是否有维度错误
    print(f"最大绝对误差: {np.max(np.abs(np_g - t_g)):.8e}")
# 对逐元素相减得到的矩阵误差取绝对值(np.abs),找最大误差,8位小数
    print(f"是否一致: {np.allclose(np_g, t_g, atol=1e-6)}")
# np.allclose(a, b, atol=1e-6):判断a,b是否近似相等 注意不用==(浮点数)
    compare("dW1", dW1, W1_t.grad.numpy())
    compare("db1", db1, b1_t.grad.numpy())
    compare("dW2", dW2, W2_t.grad.numpy())
    compare("db2", db2, b2_t.grad.numpy())
# 转为数组,调用函数对比

**结果**:NumPy手算梯度和PyTorch自动梯度一致,最大绝对误差<1e-8;四个梯度形状与参数形状一致

## 6. 把样本换成一个小batch
修改为:
`data = load_moons_data()`
`X = data['X_train_scaled'][0:16]`         # 从一个变成16个 
`y = data['y_train'][0:16].reshape(-1, 1)`  

**运行后**:和样本相关的量发生改变(X,Z1,A1...第一维从1变成N),与梯度,参数形状完全不变(神经系统能处理任意batch大小的关键)